# Gaussian Mixture Model (GMM) Clustering

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

In [ ]:
# Load the dataset
X = pd.read_csv('finalclusteringdataset.csv')
print(X.head())
print(X.shape)

In [ ]:
# Find optimal number of components using BIC and AIC
n_components_range = range(2, 11)
bic_scores = []
aic_scores = []
silhouette_scores = []

for n_components in n_components_range:
    gmm = GaussianMixture(n_components=n_components, random_state=42, n_init=10)
    gmm.fit(X)
    bic_scores.append(gmm.bic(X))
    aic_scores.append(gmm.aic(X))
    labels = gmm.predict(X)
    silhouette_scores.append(silhouette_score(X, labels))

# Find optimal number of components
optimal_n_bic = n_components_range[np.argmin(bic_scores)]
optimal_n_aic = n_components_range[np.argmin(aic_scores)]
optimal_n_silhouette = n_components_range[np.argmax(silhouette_scores)]

print(f'Optimal n_components (BIC): {optimal_n_bic}')
print(f'Optimal n_components (AIC): {optimal_n_aic}')
print(f'Optimal n_components (Silhouette): {optimal_n_silhouette}')

In [ ]:
# Plot model selection metrics
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(n_components_range, bic_scores, 'bo-')
plt.xlabel('Number of Components')
plt.ylabel('BIC')
plt.title('BIC vs Number of Components')
plt.axvline(optimal_n_bic, color='red', linestyle='--', label=f'Optimal: {optimal_n_bic}')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(n_components_range, aic_scores, 'go-')
plt.xlabel('Number of Components')
plt.ylabel('AIC')
plt.title('AIC vs Number of Components')
plt.axvline(optimal_n_aic, color='red', linestyle='--', label=f'Optimal: {optimal_n_aic}')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(n_components_range, silhouette_scores, 'ro-')
plt.xlabel('Number of Components')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs Number of Components')
plt.axvline(optimal_n_silhouette, color='red', linestyle='--', label=f'Optimal: {optimal_n_silhouette}')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Visualize clusters (2D projection)
from sklearn.decomposition import PCA

# Reduce to 2D for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Get cluster probabilities for coloring
max_probabilities = np.max(probabilities, axis=1)

plt.figure(figsize=(14, 6))

# Hard clustering
plt.subplot(1, 2, 1)
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', s=50, alpha=0.6)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('GMM Hard Clustering (Assignment)')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)

# Soft clustering (probability)
plt.subplot(1, 2, 2)
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=max_probabilities, cmap='RdYlGn', s=50, alpha=0.6)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('GMM Soft Clustering (Probability)')
plt.colorbar(scatter, label='Max Probability')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cluster distribution
unique, counts = np.unique(cluster_labels, return_counts=True)
plt.figure(figsize=(10, 5))
plt.bar(unique, counts, color='skyblue', edgecolor='black')
plt.xlabel('Cluster')
plt.ylabel('Number of Data Points')
plt.title('GMM Cluster Distribution')
plt.xticks(unique)
plt.grid(axis='y', alpha=0.3)
plt.show()

print('\nCluster Distribution:')
for cluster, count in zip(unique, counts):
    print(f'Cluster {cluster}: {count} samples ({count/len(X)*100:.2f}%)')

In [ ]:
# Compare different covariance types
covariance_types = ['full', 'tied', 'diag', 'spherical']
silhouette_scores_by_cov = {}

print('\nSilhouette scores by covariance type:')
for cov_type in covariance_types:
    gmm_cov = GaussianMixture(n_components=optimal_components, covariance_type=cov_type, random_state=42, n_init=10)
    labels_cov = gmm_cov.fit_predict(X)
    score = silhouette_score(X, labels_cov)
    silhouette_scores_by_cov[cov_type] = score
    print(f'{cov_type}: {score:.4f}')